## Setup / Imports e Paths

In [1]:
from Bio import SeqIO, AlignIO, Phylo, Entrez
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio.Blast import NCBIWWW, NCBIXML
from Bio.Phylo.TreeConstruction import DistanceCalculator, DistanceTreeConstructor


import csv
import os
import time
import re
import subprocess


# ---- PATHS ----
# Pasta base onde estão os ficheiros de entrada
pasta_inputs = r"C:\Users\dbran\Desktop\BioInformática\Labs"


# Lista dos ficheiros FASTA a processar
ficheiros_fasta = [
    os.path.join(pasta_inputs, "14296266.fasta"),
    os.path.join(pasta_inputs, "14296280.fasta"),
    os.path.join(pasta_inputs, "14296281.fasta"),
    os.path.join(pasta_inputs, "14296289.fasta"),
]


# Pasta onde todos os resultados vão ser guardados
pasta_resultados = os.path.join(pasta_inputs, "resultados") 
os.makedirs(pasta_resultados, exist_ok=True)


# Ficheiro FASTA final com as proteínas traduzidas
ficheiro_proteinas = os.path.join(pasta_resultados, "nossos_genes_traduzidos.faa")


# Pasta para guardar os resultados do BLAST em formato XML
pasta_blast = os.path.join(pasta_resultados, "blastp_xml")
os.makedirs(pasta_blast, exist_ok=True)


# Caminho para o executável do MUSCLE (alinhamento múltiplo)
muscle_exe = r"C:\Users\dbran\Desktop\BioInformática\Labs\muscle\muscle.exe"


# Verificação rápida da existência dos ficheiros e ferramentas
for f in ficheiros_fasta:
    print(f, "OK" if os.path.exists(f) else "MISSING")


print("Resultados:", pasta_resultados)
print("MUSCLE existe?", os.path.exists(muscle_exe))


C:\Users\dbran\Desktop\BioInformática\Labs/14296266.fasta MISSING
C:\Users\dbran\Desktop\BioInformática\Labs/14296280.fasta MISSING
C:\Users\dbran\Desktop\BioInformática\Labs/14296281.fasta MISSING
C:\Users\dbran\Desktop\BioInformática\Labs/14296289.fasta MISSING
Resultados: C:\Users\dbran\Desktop\BioInformática\Labs/resultados
MUSCLE existe? False


## Leitura e validação DNA

In [ ]:
# Função que verifica se uma sequência contém apenas caracteres válidos de DNA
def e_dna(sequencia: str) -> bool:
    return set(sequencia.upper()) <= set("ACGTN")


# Lista para guardar os registos FASTA validados como DNA
registos_dna = []


# Percorre todos os ficheiros FASTA de entrada
for caminho_fasta in ficheiros_fasta:
    # Lê um único registo FASTA
    registo = SeqIO.read(caminho_fasta, "fasta")
    
    # Normaliza a sequência (maiúsculas e remoção de espaços/linhas)
    seq_str = str(registo.seq).upper().replace(" ", "").replace("\n", "")
    
    # Valida se a sequência corresponde a DNA
    if not e_dna(seq_str):
        raise ValueError(f"{caminho_fasta} não parece DNA (há caracteres fora de A/C/G/T/N).")
    
    # Atualiza a sequência do registo com a versão limpa
    registo.seq = Seq(seq_str)
    
    # Guarda o registo validado
    registos_dna.append(registo)


# Resumo rápido: ID, comprimento da sequência e descrição de cada registo
[(r.id, len(r.seq), r.description) for r in registos_dna]


[('NC_019914.1:16482-17621',
  1140,
  'NC_019914.1:16482-17621 Staphylococcus phage StB27, complete genome'),
 ('NC_019914.1:29102-30352',
  1251,
  'NC_019914.1:29102-30352 Staphylococcus phage StB27, complete genome'),
 ('NC_019914.1:30363-32237',
  1875,
  'NC_019914.1:30363-32237 Staphylococcus phage StB27, complete genome'),
 ('NC_019914.1:38194-39930',
  1737,
  'NC_019914.1:38194-39930 Staphylococcus phage StB27, complete genome')]

## ORFs e tradução 

In [2]:
# Função que procura a melhor ORF traduzida considerando os 6 frames de leitura
# (3 no sentido direto e 3 no complemento reverso)
def melhor_traducao_orf_6frames(dna: Seq, min_aa=60):
    dna = dna.upper()
    candidatos = []


    # Avalia os dois sentidos da sequência: direto (+) e reverso complementar (-)
    for sentido, seq in [("+", dna), ("-", dna.reverse_complement())]:
        # Testa os três frames de leitura possíveis
        for frame in range(3):
            # Traduz a sequência a partir do frame atual, sem parar no primeiro stop
            prot = seq[frame:].translate(to_stop=False)
            
            # Divide a proteína traduzida em segmentos separados por codões stop
            partes = str(prot).split("*")
            for p in partes:
                # Ignora segmentos sem codão de início (M)
                if "M" not in p:
                    continue
                
                # Considera apenas a região desde o primeiro M
                p2 = p[p.find("M"):]
                
                # Guarda apenas ORFs com tamanho mínimo definido
                if len(p2) >= min_aa:
                    candidatos.append((len(p2), sentido, frame, p2))


    # Se não houver ORFs válidas, devolve None
    if not candidatos:
        return None, None, None


    # Ordena os candidatos por comprimento (do maior para o menor)
    candidatos.sort(reverse=True, key=lambda x: x[0])
    
    # Seleciona a ORF mais longa
    _, melhor_sentido, melhor_frame, melhor_aa = candidatos[0]
    return melhor_aa, melhor_sentido, melhor_frame


# Lista para guardar os registos traduzidos em proteína
registos_proteina = []


# Lista com informação resumida das ORFs encontradas
info_orf = []


# Processa cada sequência de DNA validada
for r in registos_dna:
    # Obtém a melhor tradução ORF nos 6 frames
    aa, sentido, frame = melhor_traducao_orf_6frames(r.seq, min_aa=60)
    
    # Se não for encontrada uma ORF adequada, faz tradução simples como fallback
    if aa is None:
        aa = str(r.seq.translate(to_stop=True))
        sentido, frame = "+", 0


    # Cria um SeqRecord de proteína com informação sobre sentido e frame
    registos_proteina.append(
        SeqRecord(Seq(aa), id=r.id, description=f"traducao_melhor_orf sentido={sentido} frame={frame}")
    )
    
    # Guarda um resumo com comprimentos e frame usados
    info_orf.append((r.id, len(r.seq), len(aa), sentido, frame))


# Mostra a informação resumida das ORFs
info_orf


NameError: name 'registos_dna' is not defined

## Guardar proteínas traduzidas

In [ ]:
#Guardar Proteinas Traduzidas
SeqIO.write(registos_proteina, ficheiro_proteinas, "fasta")
ficheiro_proteinas, [(p.id, len(p.seq)) for p in registos_proteina]


('C:\\Users\\dbran\\Desktop\\BioInformática\\Labs\\resultados\\nossos_genes_traduzidos.faa',
 [('NC_019914.1:16482-17621', 379),
  ('NC_019914.1:29102-30352', 416),
  ('NC_019914.1:30363-32237', 624),
  ('NC_019914.1:38194-39930', 578)])

## BLASTP + parsing (XML)

In [ ]:
# Função que executa um BLASTP no NCBI para uma proteína
def executar_blastp(registo_proteina, base_dados="nr", max_hits=10):
    # Envia a sequência ao servidor do NCBI e obtém o resultado em XML
    handle = NCBIWWW.qblast("blastp", base_dados, registo_proteina.format("fasta"), hitlist_size=max_hits)
    xml_texto = handle.read()
    handle.close()
    return xml_texto


# Função que guarda o resultado do BLAST em formato XML num ficheiro
def guardar_xml_blast(xml_texto, caminho_xml):
    with open(caminho_xml, "w", encoding="utf-8") as f:
        f.write(xml_texto)


# Função que lê um ficheiro XML de BLAST e extrai os melhores hits
def ler_top_hits(caminho_xml, n=5):
    # Lê o resultado BLAST a partir do XML
    with open(caminho_xml) as handle:
        registo_blast = NCBIXML.read(handle)


    hits = []
    # Percorre apenas os n melhores alinhamentos
    for alinhamento in registo_blast.alignments[:n]:
        hsp = alinhamento.hsps[0]  # Usa o melhor HSP de cada alinhamento
        hits.append({
            "hit_def": alinhamento.hit_def,
            "hit_id": getattr(alinhamento, "hit_id", None),
            "accession": getattr(alinhamento, "accession", None),
            "hit_length": getattr(alinhamento, "length", None),
            "tamanho_alinhamento": hsp.align_length,
            "identidades": hsp.identities,
            # Calcula a percentagem de identidade do alinhamento
            "percent_id": 100.0 * hsp.identities / hsp.align_length if hsp.align_length else None,
            "evalue": hsp.expect
        })
    return hits


['NC_019914.1:16482-17621',
 'NC_019914.1:29102-30352',
 'NC_019914.1:30363-32237',
 'NC_019914.1:38194-39930']

## Correr BLAST e guardar resultados

In [ ]:
# Dicionário para guardar os resultados BLAST por proteína
resumo_blast = {}


# Executa BLASTP para cada proteína traduzida
for registo in registos_proteina:
    # Cria um ID seguro para nome de ficheiro
    safe_id = registo.id.replace(":", "_").replace("-", "_")
    caminho_xml = os.path.join(pasta_blast, f"{safe_id}_blastp.xml")


    # Só executa BLAST se o ficheiro ainda não existir (evita pedidos repetidos)
    if not os.path.exists(caminho_xml):
        xml = executar_blastp(registo, base_dados="nr", max_hits=10)
        guardar_xml_blast(xml, caminho_xml)
        time.sleep(15)  # Pausa para respeitar as regras do NCBI


    # Lê e guarda os melhores hits do BLAST
    resumo_blast[registo.id] = ler_top_hits(caminho_xml, n=10)


# Resultados finais do BLAST organizados por ID da proteína
resultados_blast = resumo_blast


# Lista dos IDs das proteínas com resultados BLAST
list(resultados_blast.keys())


'C:\\Users\\dbran\\Desktop\\BioInformática\\Labs\\resultados\\resumo_genes_blast.csv'

## Exportar CSV resumo (genes + BLAST)

In [ ]:
# Define o caminho para o ficheiro CSV final que vai resumir genes e BLAST
out_csv = os.path.join(pasta_resultados, "resumo_genes_blast.csv")


# Abre o CSV para escrita
with open(out_csv, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    
    # Escreve o cabeçalho do CSV
    w.writerow(["gene_id", "len_nt", "len_aa", "top_hit_accession", "top_hit_def", "top_hit_evalue", "top_hit_percent_id"])
    
    # Para cada gene/processamento ORF
    for (gene_id, len_nt, len_aa, sentido, frame) in info_orf:
        hits = resultados_blast.get(gene_id, [])
        
        # Se houver hits, escreve os dados do top hit
        if hits:
            h0 = hits[0]
            w.writerow([gene_id, len_nt, len_aa, h0.get("accession"), h0.get("hit_def"), h0.get("evalue"), h0.get("percent_id")])
        else:
            # Caso não haja hits, escreve None nas colunas correspondentes
            w.writerow([gene_id, len_nt, len_aa, None, None, None, None])


# Retorna o caminho do CSV criado
out_csv


Guardado: C:\Users\dbran\Desktop\BioInformática\Labs\resultados\para_filogenia_com_homologos.faa
Total seqs: 44


## Preparar FASTA para filogenia (Entrez + homólogos)

In [ ]:
# Define o email para usar com Entrez (necessário para acessar o NCBI)
Entrez.email = "dbrandao2004@gmail.com"


# Parâmetros para filtrar hits BLAST
top_hits_por_gene = 10       # máximo de hits por gene
evalue_max = 1e-20           # e-value máximo permitido
min_percent_id = 30.0        # identidade mínima em %


# Ficheiro FASTA final para filogenia, incluindo os homólogos
fasta_saida = os.path.join(pasta_resultados, "para_filogenia_com_homologos.faa")


# Extrai o accession de uma string hit_def usando regex
def extrair_accession_de_hitdef(hit_def: str):
    m = re.search(r"\b[A-Z]{1,4}\d{4,}\.\d+\b", hit_def)
    return m.group(0) if m else None


# Faz fetch das sequências de proteínas a partir dos accessions
def fetch_proteinas_fasta(accessions):
    with Entrez.efetch(db="protein", id=",".join(accessions), rettype="fasta", retmode="text") as h:
        return list(SeqIO.parse(h, "fasta"))


# Lê as proteínas próprias traduzidas
meus_recs = list(SeqIO.parse(ficheiro_proteinas, "fasta"))


# Seleciona os accessions dos top hits filtrando por e-value e percentagem de identidade
accessions_por_gene = {}
todos_accessions = []


for gene_id, hits in resultados_blast.items():
    escolhidos = []
    for h in hits:
        ev = h.get("evalue", None)
        pid = h.get("percent_id", None)


        if ev is not None and ev > evalue_max:
            continue
        if pid is not None and pid < min_percent_id:
            continue


        acc = h.get("accession") or extrair_accession_de_hitdef(h.get("hit_def", ""))
        if not acc:
            continue


        if acc not in escolhidos:
            escolhidos.append(acc)


        if len(escolhidos) >= top_hits_por_gene:
            break


    accessions_por_gene[gene_id] = escolhidos
    todos_accessions.extend(escolhidos)


# Remove accessions duplicados mantendo a ordem
seen = set()
todos_accessions = [a for a in todos_accessions if not (a in seen or seen.add(a))]


# Faz fetch em lotes de 50 para evitar sobrecarga no servidor NCBI
homologos_recs = []
batch = 50
for i in range(0, len(todos_accessions), batch):
    bloco = todos_accessions[i:i+batch]
    homologos_recs.extend(fetch_proteinas_fasta(bloco))
    time.sleep(0.4)  # pequena pausa para não sobrecarregar o NCBI


# Função para encurtar descrições longas
def encurtar(txt, n=60):
    txt = re.sub(r"\s+", " ", txt).strip()
    return (txt[:n] + "...") if len(txt) > n else txt


# Combina os homólogos com IDs ajustados e descrições encurtadas
homologos_final = []
for gene_id, accs in accessions_por_gene.items():
    for acc in accs:
        rec = None
        for r in homologos_recs:
            if r.id == acc or r.id.startswith(acc):
                rec = r
                break
        if rec is None:
            continue


        rec2 = rec[:]
        rec2.id = f"{gene_id}|{rec.id}"
        rec2.description = encurtar(rec.description)
        homologos_final.append(rec2)


# Prepara também as próprias sequências do utilizador com prefixo MEU
todos_recs = []
for r in meus_recs:
    r2 = r[:]
    r2.id = f"MEU|{r.id}"
    r2.description = encurtar(r.description)
    todos_recs.append(r2)


# Junta homólogos e sequências próprias
todos_recs.extend(homologos_final)


# Escreve todas as sequências finais num FASTA para filogenia
SeqIO.write(todos_recs, fasta_saida, "fasta")
print("Guardado:", fasta_saida)
print("Total seqs:", len(todos_recs))


('C:\\Users\\dbran\\Desktop\\BioInformática\\Labs\\resultados\\genbank\\NC_019914.1.gb',
 'NC_019914.1',
 'Staphylococcus phage StB27, complete genome')

## GenBank: anotar features por intervalo

In [ ]:
# Função que extrai accession e intervalo de um ID no formato "accession:start-end"
def parse_intervalo(gene_id):
    m = re.match(r"^([^:]+):(\d+)-(\d+)$", gene_id)
    if not m:
        return None
    return m.group(1), int(m.group(2)), int(m.group(3))


# Função que faz download do genoma em GenBank pelo accession e guarda localmente
def fetch_genbank_genoma(accession, pasta_out):
    os.makedirs(pasta_out, exist_ok=True)  # cria a pasta de saída se não existir
    out = os.path.join(pasta_out, f"{accession}.gb")
    
    # Se o ficheiro já existir, retorna o caminho sem fazer download
    if os.path.exists(out):
        return out
    
    # Download do GenBank via Entrez
    with Entrez.efetch(db="nucleotide", id=accession, rettype="gb", retmode="text") as h:
        txt = h.read()
    with open(out, "w", encoding="utf-8") as f:
        f.write(txt)
    
    time.sleep(0.4)  # pausa para não sobrecarregar o NCBI
    return out


# Exemplo: extrai o accession do primeiro registo de DNA
genoma_acc = parse_intervalo(registos_dna[0].id)[0]


# Faz download e guarda o ficheiro GenBank
gb_path = fetch_genbank_genoma(genoma_acc, os.path.join(pasta_resultados, "genbank"))


# Lê o registo GenBank
record = SeqIO.read(gb_path, "genbank")


# Mostra caminho do ficheiro, ID do genoma e descrição
gb_path, record.id, record.description



== NC_019914.1:16482-17621 ==
CDS 16481 - 17621 | gene: ? | product: terminase large subunit | protein_id: YP_007236598.1
CDS 17621 - 19073 | gene: ? | product: portal protein | protein_id: YP_007236599.1

== NC_019914.1:29102-30352 ==
CDS 29101 - 30352 | gene: ? | product: tail protein with endopeptidase domain | protein_id: YP_007236612.1

== NC_019914.1:30363-32237 ==
CDS 30362 - 32237 | gene: ? | product: zinc carboxypeptidase | protein_id: YP_007236613.1

== NC_019914.1:38194-39930 ==
CDS 38193 - 39930 | gene: ? | product: endolysin | protein_id: YP_007236621.1


## Features intersect + print

In [ ]:
# Função que retorna todas as features CDS de um genoma que intersectam um intervalo específico
def features_que_intersectam(record, inicio, fim):
    feats = []
    for ft in record.features:
        if ft.type != "CDS":  # só considera CDS (códigos de proteínas)
            continue
        if ft.location is None:
            continue
        s = int(ft.location.start)
        e = int(ft.location.end)
       
        # Verifica se há interseção com o intervalo [inicio, fim]
        if not (e < inicio or s > fim):
            feats.append((s, e, ft.qualifiers))
    return feats


# Para cada gene/registo DNA, encontra e imprime as CDS que intersectam o seu intervalo
for gene_id in [r.id for r in registos_dna]:
    acc, ini, fim = parse_intervalo(gene_id)
    feats = features_que_intersectam(record, ini, fim)
    
    print("\n==", gene_id, "==")
    for s, e, q in feats[:3]:  # mostra apenas as 3 primeiras para simplificar
        print("CDS", s, "-", e,
              "| gene:", q.get("gene", ["?"])[0],
              "| product:", q.get("product", ["?"])[0],
              "| protein_id:", q.get("protein_id", ["?"])[0])


'C:\\Users\\dbran\\Desktop\\BioInformática\\Labs\\resultados\\features_genbank_por_gene.csv'

## CSV features

In [ ]:
# Define o caminho para o CSV que vai guardar as features CDS por gene
out_feat = os.path.join(pasta_resultados, "features_genbank_por_gene.csv")


# Abre o CSV para escrita
with open(out_feat, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    
    # Cabeçalho do CSV
    w.writerow(["gene_id", "cds_start", "cds_end", "gene", "product", "protein_id"])


    # Para cada gene/registo DNA
    for gene_id in [r.id for r in registos_dna]:
        acc, ini, fim = parse_intervalo(gene_id)
        
        # Obtém as CDS do GenBank que intersectam o intervalo do gene
        feats = features_que_intersectam(record, ini, fim)
        
        # Escreve cada CDS no CSV
        for s, e, q in feats:
            w.writerow([
                gene_id,
                s, e,
                q.get("gene", ["?"])[0],
                q.get("product", ["?"])[0],
                q.get("protein_id", ["?"])[0]
            ])


# Retorna o caminho do CSV criado
out_feat


NameError: name 'fasta_saida' is not defined

## MUSCLE + alinhamento + árvore

In [ ]:
# Caminho para o ficheiro FASTA do alinhamento múltiplo
ficheiro_alinhamento = os.path.join(pasta_resultados, "para_filogenia_com_homologos.aln.fasta")


# Executa MUSCLE para alinhar as sequências do FASTA
subprocess.run([muscle_exe, "-in", fasta_saida, "-out", ficheiro_alinhamento], check=True)


# Lê o alinhamento resultante
alinhamento = AlignIO.read(ficheiro_alinhamento, "fasta")
print("Alinhamento:", alinhamento.get_alignment_length(), "posições |", len(alinhamento), "seqs")


# Construção de árvore filogenética usando o método Neighbor-Joining
calculadora = DistanceCalculator("identity")            # calcula matriz de distâncias baseada na identidade
matriz_distancias = calculadora.get_distance(alinhamento)


construtor = DistanceTreeConstructor()                   # inicializa construtor de árvores
arvore = construtor.nj(matriz_distancias)              # gera árvore NJ


# Guarda a árvore em formato Newick
ficheiro_arvore = os.path.join(pasta_resultados, "arvore_com_homologos.nwk")
Phylo.write(arvore, ficheiro_arvore, "newick")


# Mostra a árvore em ASCII no terminal
Phylo.draw_ascii(arvore)


# Retorna o caminho do ficheiro da árvore
ficheiro_arvore
